<a href="https://colab.research.google.com/github/huimouqianxiao123/Peft-Qwen2.5-/blob/main/QLora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers accelerate bitsandbytes huggingface_hub peft datasets scikit-learn tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 137.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 44.9 MB/s eta 0:00:00
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.67.3
    Uninstalling tqdm-4.67.3:
      Successfully uninstalled tqdm-4.67.3
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully unins

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model_path = "/content/drive/MyDrive/models/qwen2.5-7b-instruct"
file_path = "/content/drive/MyDrive/data/intent_train_multiturn_qwen.jsonl"
adapter_output_dir = "/content/drive/MyDrive/models/qwen2.5-7b-intent-qlora"

train_random_seed = 42
eval_random_seed = 2026
max_seq_length = 1024

In [5]:
import json
import random
from datasets import Dataset

all_data = []
skipped_rows = 0

print(f"正在读取数据集: {file_path}")
with open(file_path, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        if not line.strip():
            continue

        item = json.loads(line)
        messages = item.get("messages", [])
        if len(messages) < 2 or messages[-1].get("role") != "assistant" or "content" not in messages[-1]:
            skipped_rows += 1
            continue

        all_data.append(item)

if not all_data:
    raise ValueError("没有读到可训练样本，请检查 file_path 和 JSONL 数据格式。")

train_data = all_data
train_sample_size = min(2500, len(train_data))
random.seed(train_random_seed)
sampled_train_data = random.sample(train_data, train_sample_size)
train_dataset = Dataset.from_list(sampled_train_data)

print(f"可用样本数: {len(all_data)}")
print(f"跳过异常样本数: {skipped_rows}")
print(f"本次训练随机抽取样本数: {train_sample_size}")

正在读取数据集: /content/drive/MyDrive/data/intent_train_multiturn_qwen.jsonl
可用样本数: 5000
跳过异常样本数: 0
本次训练随机抽取样本数: 2500


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


In [8]:
def tokenize_chat_example(item):
    messages = item["messages"]
    prompt_messages = messages[:-1]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    tokenized = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_seq_length,
    )

    input_ids = tokenized["input_ids"]
    labels = input_ids.copy()
    prompt_length = min(len(prompt_ids), len(labels))
    labels[:prompt_length] = [-100] * prompt_length

    return {
        "input_ids": input_ids,
        "attention_mask": tokenized["attention_mask"],
        "labels": labels,
    }


tokenized_train_dataset = train_dataset.map(
    tokenize_chat_example,
    remove_columns=train_dataset.column_names,
    desc="正在构建 QLoRA 训练样本",
)

正在构建 QLoRA 训练样本:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [18]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

training_args = TrainingArguments(
    output_dir=f"{adapter_output_dir}/trainer_outputs",
    num_train_epochs=3,
    per_device_train_batch_size=10,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained(adapter_output_dir)
tokenizer.save_pretrained(adapter_output_dir)
print(f"QLoRA adapter 已保存到: {adapter_output_dir}")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
10,0.092767
20,0.071666
30,0.084894
40,0.043462
50,0.025592
60,0.029854
70,0.016348
80,0.011853
90,0.008242


QLoRA adapter 已保存到: /content/drive/MyDrive/models/qwen2.5-7b-intent-qlora


In [19]:
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

intents_list = [
    "查询物流", "催促配送", "修改订单", "取消订单", "申请退款", "退换货",
    "发票咨询", "支付咨询", "价格价保", "优惠活动", "商品咨询", "催促处理",
    "投诉反馈", "转人工", "问候确认", "业务咨询", "查询订单", "其他",
]


def normalize_label(response_text):
    text = response_text.strip()
    for intent in intents_list:
        if text == intent or intent in text:
            return intent
    return "其他"


eval_sample_size = min(500, len(all_data))
random.seed(eval_random_seed)
eval_dataset = random.sample(all_data, eval_sample_size)

model.eval()
y_true = []
y_pred = []
raw_outputs = []

print(f"开始随机评估 {eval_sample_size} 条样本，并逐条打印结果...")
for idx, item in enumerate(tqdm(eval_dataset), start=1):
    messages = item.get("messages", [])
    prompt_messages = messages[:-1]
    true_label = messages[-1]["content"].strip()

    if true_label not in intents_list:
        continue

    text = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(


        [text],
        return_tensors="pt",
        truncation=True,
        max_length=max_seq_length,
    ).to(model.device)

    with torch.inference_mode():
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    output_ids = generated_ids[0][model_inputs.input_ids.shape[-1]:]
    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()
    pred_label = normalize_label(response)
    is_correct = pred_label == true_label

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append({
        "index": idx,
        "true_label": true_label,
        "pred_label": pred_label,
        "raw_response": response,
        "is_correct": is_correct,
        "current_user_utterance": item.get("metadata", {}).get("current_user_utterance", ""),
    })

    result_row = (
        f"{idx:03d} | 当前话语: {raw_outputs[-1]['current_user_utterance']} | "
        f"真实: {true_label} | 预测: {pred_label} | "
        f"原始输出: {response} | {'正确' if is_correct else '错误'}"
    )
    print(result_row)

if not y_true:
    raise ValueError("没有有效评估结果，请检查标签集合或数据格式。")

accuracy = accuracy_score(y_true, y_pred)

print("\n================== 微调模型测试评估报告 ==================")
print(f"训练随机样本数: {train_sample_size}")
print(f"训练轮数: 10")
print(f"有效评估样本数: {len(y_true)}")
print(f"总体准确率 (Overall Accuracy): {accuracy:.2%}\n")
print("各意图类别的详细指标 (Precision / Recall / F1-Score):")
print(classification_report(y_true, y_pred, labels=intents_list, zero_division=0))
print("=========================================================")

开始随机评估 500 条样本，并逐条打印结果...


  0%|          | 0/500 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


001 | 当前话语: 没有了 | 真实: 问候确认 | 预测: 问候确认 | 原始输出: 问候确认 | 正确
002 | 当前话语: 谢谢 | 真实: 问候确认 | 预测: 问候确认 | 原始输出: 问候确认 | 正确
003 | 当前话语: 先买，送货然后把退的给服务么? | 真实: 查询物流 | 预测: 查询物流 | 原始输出: 查询物流 | 正确
004 | 当前话语: 好的 | 真实: 问候确认 | 预测: 问候确认 | 原始输出: 问候确认 | 正确
005 | 当前话语: 顾客通过点击web咚咚[站点x]信息发送:[订单编号:[ORDERID_10002568]，订单金额:[金额x]，下单时间:[日期x] [时间x]] | 真实: 查询物流 | 预测: 查询物流 | 原始输出: 查询物流 | 正确
006 | 当前话语: 大冷天的，添麻烦了。。。 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
007 | 当前话语: 我已经申请换货了， 还可以申请售后吗 | 真实: 退换货 | 预测: 退换货 | 原始输出: 退换货 | 正确
008 | 当前话语: 评价好了，多休息，再见 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
009 | 当前话语: 时间基本正确啊 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
010 | 当前话语: 帮我取消就好了 | 真实: 取消订单 | 预测: 其他 | 原始输出: 其他 | 错误
011 | 当前话语: 是的 | 真实: 问候确认 | 预测: 问候确认 | 原始输出: 问候确认 | 正确
012 | 当前话语: 订单显示签收 | 真实: 查询物流 | 预测: 查询物流 | 原始输出: 查询物流 | 正确
013 | 当前话语: 我买这自营的裤子才穿几次就起球了 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
014 | 当前话语: 好的，麻烦了 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
015 | 当前话语: 我[数字x]号就要走，如果[数字x]号送不到的话，我改一下收货地址。 | 真实: 其他 | 预测: 其他 | 原始输出: 其他 | 正确
016 | 当前话语: 好的，麻烦您也帮忙备注下不取消订单了 | 